# 03 Run Experiments

Use this notebook when you want to run models interactively instead of using the command line scripts. The default settings are intentionally smaller than the full matrix, but you can flip `RUN_FULL_MATRIX` to `True`.

In [ ]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f'Project root: {PROJECT_ROOT}')

In [ ]:
from itertools import product

import pandas as pd
from IPython.display import display

from src.evaluation.experiment_runner import run_single_experiment
from src.utils.io import load_yaml
from src.utils.seeds import set_global_seed

## Choose Matrix Size

In [ ]:
config = load_yaml('configs/experiment_config.yaml')

RUN_FULL_MATRIX = False
OUTPUT_PATH = Path('results/tables/notebook_experiment_results.csv')

if RUN_FULL_MATRIX:
    datasets = config['datasets']
    noise_types = config['noise_types']
    noise_rates = config['noise_rates']
    models = config['models']
    detectors = config['detectors']
    seeds = config['seeds']
else:
    datasets = ['breast_cancer', 'wine']
    noise_types = ['symmetric', 'class_dependent']
    noise_rates = [0.1, 0.3]
    models = ['logistic_regression', 'random_forest']
    detectors = ['combined', 'confidence']
    seeds = [42, 123]

cv_folds = int(config.get('cv_folds', 5))

n_combos = len(datasets) * len(noise_types) * len(noise_rates) * len(models) * len(seeds)
rows_per_combo = 2 + 2 * len(detectors)
print(f'Combinations: {n_combos}')
print(f'Expected rows: {n_combos * rows_per_combo}')
print(f'Output: {OUTPUT_PATH}')

## Run and Save

In [ ]:
all_results = []

for dataset_name, noise_type, noise_rate, model_name, seed in product(
    datasets,
    noise_types,
    noise_rates,
    models,
    seeds,
):
    set_global_seed(int(seed))
    print(
        'Running '
        f'dataset={dataset_name}, '
        f'noise_type={noise_type}, '
        f'noise_rate={noise_rate}, '
        f'model={model_name}, '
        f'seed={seed}'
    )
    result = run_single_experiment(
        dataset_name=dataset_name,
        noise_type=noise_type,
        noise_rate=float(noise_rate),
        model_name=model_name,
        random_state=int(seed),
        cv_folds=cv_folds,
        detector_methods=detectors,
    )
    all_results.append(result)

results = pd.concat(all_results, ignore_index=True)
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
results.to_csv(OUTPUT_PATH, index=False)
print(f'Saved {OUTPUT_PATH}')
display(results.head())

## Notebook Summary Table

In [ ]:
group_cols = ['dataset', 'noise_type', 'noise_rate', 'model', 'condition', 'detector']

summary = (
    results.groupby(group_cols, dropna=False)
    .agg(
        accuracy_mean=('accuracy', 'mean'),
        accuracy_std=('accuracy', 'std'),
        f1_macro_mean=('f1_macro', 'mean'),
        f1_macro_std=('f1_macro', 'std'),
        auroc_mean=('auroc', 'mean'),
        det_f1_mean=('det_f1', 'mean'),
    )
    .reset_index()
)

summary_path = OUTPUT_PATH.with_name(OUTPUT_PATH.stem + '_summary.csv')
summary.to_csv(summary_path, index=False)
print(f'Saved {summary_path}')
display(summary.sort_values('accuracy_mean', ascending=False).head(20))